# Chapter 4: Advanced Topics, Use Cases and Additional Learning Resources

Welcome to Chapter 4, the final chapter of the course 5 minutes to Federated Learning with NVIDIA FLARE!

In [chapter 2](Chapter_2_Develop_Federated_Application.ipynb) and [chapter 3](Chapter_3_Provision_Run_and_Monitor_Federated_Project.ipynb), we learned how to develop federated applications using NVIDIA FLARE's APIs, run them in a simulated environment, as well as how to provision a federated system to run federated applications in NVIDIA FLARE's proof-of-concept mode and monitor runtime metrics. While these are foundational skills to implement federated projects, there are more aspects in NVIDIA FLARE to bring federated projects to real-world production at scale. 

In this final chapter of the course, we first aim to introduce some advanced topics in NVIDIA FLARE for real-world deployment. These advanced topics include:  
- Security: privacy preserving technologies, site policy management, Confidential Compute etc.
- FLARE Dashboard
- Deployment Options
- Flower-on-FLARE

Then, we will walk through typical use cases and real-world deployment of federated applications and projects leveraging NVIDIA FLARE. To finish up, we will share references to additional learning resources for you to become a full-fledged federated learning developer.

In this chapter we will only provide general overviews, without hands-on coding and exercises.

After this chapter, you will:
- Have a high-level understanding of FLARE's advanced features for real-world deployment of federated applications. 
- Know how and where to search and find guidelines to use or implement FLARE's advanced features.
- Get a sense of typical use-cases and real-world applications leveraging NVIDIA FLARE
- Have access to all the learning resources if you aim to master NVIDIA FLARE or maybe become a FLARE developer.


# Security

Security is an essential element in real-world deployment of federated application. NVIDIA FLARE provides many advanced [security features](https://nvidia.github.io/NVFlare/security/) to ensure that a federated application can be bullet-proof in regard to attacks and information leakage.

Below, we briefly introduce privacy preserving technologies, site policy management and Confidential Computing in FLARE.

## Privacy Preserving Technologies

[Privacy-preserving technologies](https://en.wikipedia.org/wiki/Privacy-enhancing_technologies) are technologies that embody fundamental data protection principles by minimizing personal data use, maximizing data security, and protecting individuals. In federated learning, various privacy preserving methods can be employed to protect sensitive data while enabling collaborative model training. Commonly used methods include [differential privacy](https://en.wikipedia.org/wiki/Differential_privacy), [homomorphic encryption](https://en.wikipedia.org/wiki/Homomorphic_encryption) etc.

In NVIDIA FLARE, most privacy preserving methods are implemented leveraging the powerful [filtering mechanism](https://nvflare.readthedocs.io/en/main/programming_guide/filters.html#filters), as shown in the figure below (remember the FLARE architecture diagram in [chapter 2](Chapter_2_Develop_Federated_Application.ipynb#NVIDIA-FLARE-Architecture) that shows task filtering mechanism?). As a matter of fact, the filtering mechanism in FLARE offers general [data privacy protection](https://nvflare.readthedocs.io/en/main/user_guide/security/data_privacy_protection.html), with flexibility to add any filtering to any moment of inbound and outbound task data exchange between both the server Controllers and client executors. While FLARE provides reference implementations of privacy preserving filters, you can also [write your own custom filters leveraging FLARE's Data Exchange Object class](https://nvflare.readthedocs.io/en/main/programming_guide/filters.html#creating-a-dxo-filter).

<img src="../images/filtering.png" alt="Filtering" width=50% />

Below, we briefly describe differential privacy and homomorphic encryption in FLARE. Refer to this [security page](https://nvidia.github.io/NVFlare/security/) for more details.

### Differential Privacy

[Differential Privacy](https://en.wikipedia.org/wiki/Differential_privacy) is a mathematically rigorous framework for protecting individual data privacy while allowing statistical analysis of sensitive data. Common differential privacy algorithms are essentially adding controlled noise to data or results. They ensure that the inclusion or exclusion of an individual's data does not significantly affect the results of analyses. Differential privacy offers strong privacy guarantees while still allowing useful insights to be drawn from data, making it a powerful tool for privacy-preserving data analysis. 

FLARE provides a reference implementation of [differential privacy filter using the Sparse Vector Technique](https://nvflare.readthedocs.io/en/main/apidocs/nvflare.app_common.filters.svt_privacy.html). You can also customise your own differential privacy filter.

You can refer to [this example](https://github.com/NVIDIA/NVFlare/tree/main/examples/advanced/brats18) to learn how to use differential privacy in FLARE.

### Homomorphic Encryption

[Homomorphic encryption](https://en.wikipedia.org/wiki/Homomorphic_encryption) is a cryptographic method that allows computations to be performed on encrypted data without decrypting it first. It enables mathematical operations on cipher text, producing encrypted results that, when decrypted, match the results of performing the same operations on the original plain text. Homomorphic encryption allows for secure aggregation of model updates and protects data privacy during transmission and aggregation.

In NVIDIA FLARE, similar to differential privacy, homomorphic encryption is implemented as a filter, to [encrypt data](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_opt/he/model_encryptor.py#L33) before sharing and to [decrypt result](https://github.com/NVIDIA/NVFlare/blob/main/nvflare/app_opt/he/model_decryptor.py#L35) after compute. Internally, NVIDIA FLARE uses the [TenSEAL](https://github.com/OpenMined/TenSEAL) library, which is a Python wrapper around [Microsoft SEAL](https://github.com/Microsoft/SEAL), for implementing homomorphic encryption.

You can find many examples illustrating how to use homomorphic encryption with NVIDIA FLARE, for instance, [here](https://github.com/NVIDIA/NVFlare/blob/main/examples/hello-world/step-by-step/cifar10/sag_he/sag_he.ipynb) and [here](https://github.com/NVIDIA/NVFlare/blob/main/examples/advanced/cifar10/cifar10-real-world/README.md).

Privacy preserving filters can be enforced to the server or clients at job-level using the FedJob API. For instance, see [this example](https://github.com/NVIDIA/NVFlare/blob/main/examples/advanced/job_api/pt/fedavg_script_runner_dp_filter_cifar10.py) which adds differential privacy filters to clients for task results. Filters can also be applied at site-level, with [site-specific configuration](https://nvflare.readthedocs.io/en/main/user_guide/security/site_policy_management.html#privacy-management). We will see more details on this in the next section.

## Site Policy Management

<img src="../images/local-config.png" alt="Site Policy Management" width=40% />

As we have seen together in [chapter 3](Chapter_3_Provision_Run_and_Monitor_Federated_Project.ipynb#2.-Generate-startup-kits), inside each site's startup kit, there is a `local` folder with default site-specific configurations (see figure above). These configurations can be customized allowing each site to define its own policies in the following areas:
- Authorization Policy: local authorization policy that determines what a user can or cannot do on the local site.
- Logging Configuration: each site can define its own logging configuration for system generated log messages.
- Privacy Policy: local policy that specifies what types of studies are allowed and how to add privacy protection to the learning results produced by the clients on the local site.
- Resource Management: the configuration of system resources that are solely the decisions of local IT.

For more details on site policy management, refer to the [dedicated documentation page](https://nvflare.readthedocs.io/en/main/user_guide/security/site_policy_management.html).

## Confidential Computing

[Confidential computing (CC)](https://en.wikipedia.org/wiki/Confidential_computing) is an advanced technique to protect data while it's in-use, by performing computations in a hardware-based, attested Trusted Execution Environment. At its core, the technology creates an isolated, encrypted computing environment within a processor that prevents unauthorized access or modification of data and applications while they are being processed. Modern CC implementations provide hardware-level isolation through specialized processor features, leveraging cryptographic techniques to establish a secure perimeter around computational workloads, creating a "black box" where sensitive operations occur. The figure below depicts a typical compute node with NVIDIA GPU and CC.

<img src="../images/cc.png" alt="Confidential Computing" width=25% />

The attestation process is a critical component of CC. It provides cryptographic evidence that the computing environment is genuinely secure and has not been compromised. This allows organizations to verify the integrity of the computational environment before transmitting sensitive data. By protecting data during computation, CC addresses a significant security gap that traditional encryption methods couldn't resolve, offering unprecedented levels of data protection across distributed computing environments.

Confidential computing in NVIDIA FLARE is designed to explicitly establish trust between participants. Each participant must first capture the evidences related to the hardware (such as GPU), the software (for instance GPU driver and VBIOS) and other components in its own platform. The evidences, bundled in a confidential computing token (CC token), will be validated and signed to ensure their validity and authenticity. The owner of signed evidences can demonstrate the information about its computing environment to other participants by providing the CC token. Upon receiving the CC token, the participant (the relying party) can verify the claims inside the CC token against its own security policy on whether the CC token owner is using required hardware/software/components for security. If the relying party finds that the CC token does not meet its security policy, the relying party can inform the system that it chooses not to join the job deployment and will not exchange models with others. Only participants who trust and are trusted by one another will work together to run federated jobs.

For more details on CC in FLARE, refer to this [dedicated documentation page](https://nvflare.readthedocs.io/en/main/user_guide/confidential_computing.html). You can also learn more about GPU-based CC in FLARE by watching [this video](https://developer.download.nvidia.com/assets/Clara/flare/NVFLARE_DAY_2024_Part_13_Confidential_Computing_Closing.mp4).

There are many other security features in NVIDIA FLARE. For more details, please refer to this dedicated [security documentation](https://nvflare.readthedocs.io/en/main/user_guide/nvflare_security.html).

# FLARE Dashboard

As we have seen in [chapter 3](Chapter_3_Provision_Run_and_Monitor_Federated_Project.ipynb#Provisioning-in-NVIDIA-FLARE), there are two ways to provision a project in FLARE, one is using the `nvflare provision` command, and the other is using [FLARE Dashboard](https://nvflare.readthedocs.io/en/main/user_guide/dashboard_ui.html#nvflare-dashboard-ui).

FLARE Dashboard is a web user interface (UI) that helps the project administrator to deploy a website to gather information about sites, users and distribute startup kits. As mentioned in [chapter 3](Chapter_3_Provision_Run_and_Monitor_Federated_Project.ipynb#Provisioning-in-NVIDIA-FLARE) during the Provisioning section, FLARE system requires a set of startup kits which include the private keys and certificates for participants to communicate to one another. FLARE Dashboard UI in NVIDIA FLARE provides a simple way to collect and distribute the information of participants from different organizations, as well as to generate those startup kits for users to download.

<img src="../images/dashboard.png" alt="Dashboard" width=50% />

Setting up FLARE Dashboard for a federated project allows users to register to join the project and provide their own information, and then download their own startup kits once the project admin has approved the registration. All the project information can be managed online with provisioning done on the fly.

Different administrators with different [roles](https://nvflare.readthedocs.io/en/main/user_guide/security/terminologies_and_roles.html#role) can leverage FLARE Dashboard to easily manage and register user accounts, download and distribute startup kits or specify the name and resource specifications for client sites.

FLARE Dashboard runs in a dedicated docker environment, and can be set up and managed using `nvflare dashboard` CLI tool. FLARE Dashboard also offers [backend RESTful APIs](https://nvflare.readthedocs.io/en/main/user_guide/dashboard_api.html#nvidia-flare-dashboard-backend-api) allowing for programmatic management. For information about how to use and manage FLARE Dashboard, refer to the [documentation page](https://nvflare.readthedocs.io/en/main/user_guide/dashboard_api.html#). For a detailed walkthrough of FLARE Dashboard's UI for different user types, refer to [this tutorial](https://nvflare.readthedocs.io/en/main/user_guide/dashboard_ui.html#nvflare-dashboard-ui). 


# Deployment Options

Here we briefly describe deployment options in NVIDIA FLARE.

## On-Premise Deployment

We've already seen simulated local federated project deployment using either [FL simulator](https://nvflare.readthedocs.io/en/main/user_guide/nvflare_cli/fl_simulator.html) or the [proof-of-concept mode](https://nvflare.readthedocs.io/en/main/user_guide/nvflare_cli/poc_command.html). In production mode, you can deploy projects on-premise after [provisioning](https://nvflare.readthedocs.io/en/main/real_world_fl/overview.html#provision) either with `nvflare provision` CLI tool or the FLARE Dashboard. 

NVIDIA FLARE offers more options for real-world project deployment.

## Cloud Deployment

NVIDIA FLARE supports deployment on major cloud platforms such as Azure and AWS. Cloud deployment can be enabled for a participant when launching its startup script by specifying the `--cloud` option. Addditionally, FLARE Dashboard can also be deployment in the cloud. Read [this dedicated documentation](https://nvflare.readthedocs.io/en/main/real_world_fl/cloud_deployment.html) for more details on cloud deployment.

## Containerized Deployment

FLARE supports deployment in `docker compose` mode. The provisioning tool of NVIDIA FLARE includes a `DockerBuilder` that can create `compose.yaml` and other information during provisioning, after which, users can `docker compose build` and `docker compose up` to start the servers and clients in the `docker compose` manner. Read the [dedicated documentation](https://nvflare.readthedocs.io/en/main/user_guide/docker_compose.html) for more details on `docker compose` deployment.

Sometimes, users would like to deploy NVIDIA FLARE to an existing Kubernetes cluster. FLARE provides the `HelmChartBuilder` that can generate a reference Helm Chart for users to deploy the provisioned FL project to a local Kubernetes instance. Read the [documentation here](https://nvflare.readthedocs.io/en/main/user_guide/helm_chart.html) for more details on Kubernetes deployment. FLARE also provides [recipes for deployment on the Amazon Elastic Kubernetes Service](https://nvflare.readthedocs.io/en/main/real_world_fl/kubernetes.html).


# Flower-on-FLARE

NVIDIA FLARE provides integration with Flower library to enhance the federated learning ecosystem. 

<img src="../images/flower.png" alt="Flower Integration" width=30% />

[Flower](https://flower.ai/) is an open-source project that implements a unified approach to federated learning, analytics, and evaluation. Flower has developed a large set of strategies and algorithms for FL application development, as well as a healthy FL research community. FLARE, complementary to Flower, provides an enterprise-ready, robust runtime environment for real-world federated applications.

With the Flower-on-FLARE integration, applications developed with the Flower framework will run easily in FLARE runtime without the need to make any changes. All you need to do is to configure the Flower application into a FLARE job and submit the job to the FLARE system.

For more details regarding the design of Flower-on-FLARE, refer to the dedicated [documentation page](https://nvflare.readthedocs.io/en/main/user_guide/flower_integration.html#) and [this blog post](https://flower.ai/blog/2024-03-15-announcing-nvidia-and-flower-collaboration/). You can also refer to [this video](https://developer.download.nvidia.com/assets/Clara/flare/NVFLARE_DAY_2024_Part_06_Flower.mp4) to learn more about Flower-on-FLARE. FLARE also provides an [example](https://github.com/NVIDIA/NVFlare/tree/main/examples/hello-world/hello-flower) showing how to leverage Flower-on-FLARE in action.

# Use Cases and Real-world Examples

NVIDIA FLARE has been used to implement many real-world federated use cases and examples. Here is a brief list of use cases and examples with links to more details.

## Healthcare
  - [Federated Learning for Prostate Segmentation from Multi-source Data](https://github.com/NVIDIA/NVFlare/blob/main/examples/advanced/prostate/README.md)
  - [Federated Learning with Differential Privacy for BraTS18 Segmentation](https://github.com/NVIDIA/NVFlare/blob/dev/examples/advanced/brats18/README.md)
  - [Federated Drug Discovery with BioNemo](https://github.com/NVIDIA/NVFlare/tree/main/examples/advanced/bionemo)

## Finance
  - [Financial Application with Federated XGBoost Methods](https://github.com/NVIDIA/NVFlare/tree/main/examples/advanced/finance)

## Swarm Learning and Split Learning
  - [Swarm Learning with Cross-Site Evaluation](https://github.com/NVIDIA/NVFlare/tree/main/examples/advanced/swarm_learning)
  - [Split Learning with CIFAR-10](https://github.com/NVIDIA/NVFlare/tree/main/examples/advanced/vertical_federated_learning/cifar10-splitnn)

## Federated Large Language Models
  - [Parameter-Efficient Fine-Tuning (PEFT) with NeMo](https://github.com/NVIDIA/NVFlare/tree/main/integration/nemo/examples)
  - [Supervised Fine-tuning (SFT) with NeMo](https://github.com/NVIDIA/NVFlare/blob/main/integration/nemo/examples/supervised_fine_tuning/README.md)
  - [Prompt Learning with NeMo](https://github.com/NVIDIA/NVFlare/blob/main/integration/nemo/examples/prompt_learning/README.md)

For a comprehensive list of example use cases, refer to the [catalog here](https://nvidia.github.io/NVFlare/catalog/).


# Additional Learning Resources

Here are some additional learning resources if you are willing to go deeper into NVIDIA FLARE.

- NVIDIA FLARE's main webpage: [https://nvidia.github.io/NVFlare](https://nvidia.github.io/NVFlare). Here you can an introductory video to FLARE, as well as useful links to tutorials, blogs and research publication from FLARE teams.
- Sources code are hosted on GitHub: [https://github.com/NVIDIA/NVFlare](https://github.com/NVIDIA/NVFlare). Feel free to regularly watch the repository for [releases](https://github.com/NVIDIA/NVFlare/releases), [issues](https://github.com/NVIDIA/NVFlare/issues), [pull requests](https://github.com/NVIDIA/NVFlare/pulls) and [discussions](https://github.com/NVIDIA/NVFlare/discussions).

FLARE's [official documentation](https://nvflare.readthedocs.io/en/main/index.html) is also a good place to learn about further details including advanced features, APIs and best practices.

NVIDIA FLARE includes a comprehensive [catalog of tutorials and examples](https://nvidia.github.io/NVFlare/catalog/). You can pull these examples and run them yourself for further learning. Some of the examples and tutorials also include Jupyter notebooks.

Over recent years, FLARE's engineering and research teams have published many [technical blogs](https://developer.nvidia.com/blog/tag/federated-learning) and [research articles](https://nvflare.readthedocs.io/en/main/publications_and_talks.html), feel free to browse them to learn more.

Lastly, like other open-source projects, FLARE welcomes contributions from the open-source community. When you're ready, feel free to read the [contribution guideline](https://github.com/NVIDIA/NVFlare/blob/main/CONTRIBUTING.md) and start contributing!


# Recap

Let's recap what we've learned in this chapter:
- We had an introductory overview of some advanced topics in FLARE including security features (privacy preserving technologies, site policy management and Confidential Computing), FLARE Dashboard, and Federated LLMs.
- Then we introduced additional links to further learning resources for you to go deeper in FLARE's design, towards becoming a FLARE developer.

# Congratulations!
That's it, you have officially finished the course "5 Minutes to Federated Learning", good job!
